In [9]:
import pandas as pd
import numpy as np

In [10]:
sales = pd.read_csv("../data/raw/sales_daily.csv")

In [11]:
sales.head()

,Date,SKU,Units_Sold,Revenue,Price,Promotion
0,2024-01-01,SKU001,5,18320.25,3664.05,0
1,2024-01-01,SKU002,15,57085.35,3805.69,0
2,2024-01-01,SKU003,5,40391.15,8078.23,0
3,2024-01-01,SKU004,5,34307.85,6861.57,0
4,2024-01-01,SKU005,12,113918.64,9493.22,0


In [12]:
sales.shape

(36550, 6)

In [13]:
sales["Date"] = pd.to_datetime(sales["Date"])

In [14]:
sales.dtypes

Date          datetime64[us]
SKU                      str
Units_Sold             int64
Revenue              float64
Price                float64
Promotion              int64
dtype: object

In [15]:
sales.isnull().sum()

Date          0
SKU           0
Units_Sold    0
Revenue       0
Price         0
Promotion     0
dtype: int64

In [16]:
sales.duplicated().sum()

np.int64(0)

In [17]:
sales[sales["Units_Sold"] == 0].head(10)

,Date,SKU,Units_Sold,Revenue,Price,Promotion
135,2024-01-03,SKU036,0,0.0,5687.00,0
552,2024-01-12,SKU003,0,0.0,8078.23,0
779,2024-01-16,SKU030,0,0.0,9433.51,0
810,2024-01-17,SKU011,0,0.0,1444.56,0
824,2024-01-17,SKU025,0,0.0,11558.01,0
1188,2024-01-24,SKU039,0,0.0,7118.95,0
1253,2024-01-26,SKU004,0,0.0,6861.57,0
1860,2024-02-07,SKU011,0,0.0,1444.56,0
1988,2024-02-09,SKU039,0,0.0,7118.95,0
2060,2024-02-11,SKU011,0,0.0,1444.56,0


In [18]:
sales[sales["Units_Sold"] == 0][["Units_Sold", "Revenue", "Price", "Promotion"]].describe()

,Units_Sold,Revenue,Price,Promotion
count,202.0,202.0,202.000000,202.000000
mean,0.0,0.0,6688.690198,0.064356
std,0.0,0.0,3900.371493,0.245996
min,0.0,0.0,882.720000,0.000000
25%,0.0,0.0,1444.560000,0.000000
50%,0.0,0.0,7118.950000,0.000000
75%,0.0,0.0,11558.010000,0.000000
max,0.0,0.0,11558.010000,1.000000


In [19]:
sales[["Units_Sold", "Revenue", "Price"]].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])

,Units_Sold,Revenue,Price
0.01,1.0,1444.56,663.460
0.05,2.0,7118.95,1379.550
0.25,7.0,27643.70,3484.090
0.50,13.0,56125.90,5649.795
0.75,20.0,127954.32,8516.600
0.95,29.0,239158.15,10939.540
0.99,37.0,316057.32,11642.450


In [20]:
sales.nlargest(10, "Units_Sold")[[
    "Date", "SKU", "Units_Sold", "Revenue", "Price", "Promotion"
]]

,Date,SKU,Units_Sold,Revenue,Price,Promotion
7644,2024-06-01,SKU045,56,429899.68,7676.78,1
3494,2024-03-10,SKU045,55,422222.90,7676.78,1
21661,2025-03-09,SKU012,53,465306.61,8779.37,1
22017,2025-03-16,SKU018,51,284345.91,5575.41,1
3825,2024-03-17,SKU026,50,546977.00,10939.54,1
24286,2025-04-30,SKU037,50,137371.00,2747.42,0
4461,2024-03-30,SKU012,49,430189.13,8779.37,1
4511,2024-03-31,SKU012,49,430189.13,8779.37,1
7656,2024-06-02,SKU007,49,250590.41,5114.09,1
15311,2024-11-02,SKU012,49,430189.13,8779.37,1


In [21]:
sales.nlargest(10, "Revenue")[[
    "Date", "SKU", "Units_Sold", "Revenue", "Price", "Promotion"
]]

,Date,SKU,Units_Sold,Revenue,Price,Promotion
3825,2024-03-17,SKU026,50,546977.00,10939.54,1
3475,2024-03-10,SKU026,47,514158.38,10939.54,1
7975,2024-06-08,SKU026,47,514158.38,10939.54,1
22675,2025-03-29,SKU026,46,503218.84,10939.54,1
22325,2025-03-22,SKU026,45,492279.30,10939.54,1
27284,2025-06-29,SKU035,45,489158.55,10870.19,1
8384,2024-06-16,SKU035,43,467418.17,10870.19,1
21661,2025-03-09,SKU012,53,465306.61,8779.37,1
3425,2024-03-09,SKU026,42,459460.68,10939.54,1
22025,2025-03-16,SKU026,42,459460.68,10939.54,1


In [22]:
sales["Expected_Revenue"] = sales["Units_Sold"] * sales["Price"]

In [23]:
sales["Revenue_Difference"] = (
    sales["Revenue"] - sales["Expected_Revenue"]
)

In [24]:
sales["Revenue_Difference"].describe()

count    3.655000e+04
mean    -1.366531e-12
std      9.165989e-12
min     -5.820766e-11
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      5.820766e-11
Name: Revenue_Difference, dtype: float64

In [25]:
(sales["Revenue_Difference"].abs() > 0.01).sum()


np.int64(0)

In [26]:
clean_sales = sales.copy()

In [27]:
clean_sales = clean_sales.drop(
    columns=["Expected_Revenue", "Revenue_Difference"],
    errors="ignore"
)


In [28]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [29]:
clean_sales.to_csv(
    "../data/processed/sales_clean.csv",
    index=False
)

In [30]:
clean_sales.shape


(36550, 6)